In [ ]:
# ! pip install --upgrade --quiet google-cloud-logging google-cloud-firestore google-cloud-aiplatform langchain "langchain-experimental==0.3.4" langchain-community langchain-google-vertexai pymupdf "langchain-core<2.0.0" "langchain-text-splitters<2.0.0"

In [51]:
import vertexai
import logging
import google.cloud.logging
from vertexai.language_models import TextEmbeddingModel
from vertexai.generative_models import GenerativeModel

import pickle
from IPython.display import display, Markdown

from langchain_google_vertexai import VertexAIEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_experimental.text_splitter import SemanticChunker

from google.cloud import firestore
from google.cloud.firestore_v1.vector import Vector
from google.cloud.firestore_v1.base_vector_query import DistanceMeasure

In [ ]:
# Next, initialize Vertex AI with your project-id qwiklabs-gcp-00-fc042da96977 and a location of us-central1
vertexai.init(project="qwiklabs-gcp-00-fc042da96977", location="us-central1")

In [5]:
# prompt: Populate a variable named embedding_model with an instance of the langchain_google_vertexai class VertexAIEmbeddings. Pass it a parameter model_name set to the text embedding model version of text-embedding-005. You will use this LangChain class for your embedding model so that you can use a LangChain semantic chunker to chunk your dataset.

embedding_model = VertexAIEmbeddings(model_name="text-embedding-005")


/usr/local/lib/python3.12/dist-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [7]:
# Download the New York City Department of Health and Mental Hygiene's Food Protection Training Manual. This document will serve as your Retrieval-Augmented Generation source content.

!gcloud storage cp gs://partner-genai-bucket/genai069/nyc_food_safety_manual.pdf .


Copying gs://partner-genai-bucket/genai069/nyc_food_safety_manual.pdf to file://./nyc_food_safety_manual.pdf

Average throughput: 156.9MiB/s


In [9]:
# prompt: Use the LangChain class PyMuPDFLoader to load the contents of the PDF to a variable named data

loader = PyMuPDFLoader("/content/nyc_food_safety_manual.pdf")
data = loader.load()


In [53]:
# The following function is provided to do some basic cleaning on artifacts found in this particular document. Create a variable called cleaned_pages that is a list of strings, with each string being a page of content cleaned by this function.

def clean_page(page):
  return page.page_content.replace("-\n","")\
                          .replace("\n"," ")\
                          .replace("\x02","")\
                          .replace("\x03","")\
                          .replace("fo d P R O T E C T I O N  T R A I N I N G  M A N U A L","")\
                          .replace("N E W  Y O R K  C I T Y  D E P A R T M E N T  O F  H E A L T H  &  M E N T A L  H Y G I E N E","")

In [92]:
# prompt: Use LangChain's SemanticChunker with the embedding_model you created earlier
# to split the first five pages of cleaned_pages into text chunks.
# The SemanticChunker determines when to start a new chunk
# when it encounters a larger distance between sentence embeddings.
# Save the strings of page content
# from the resulting documents into a list of strings called chunked_content.
# Take a look at a few of the chunks to get familiar with the content.

cleaned_pages = [clean_page(page) for page in data[:5]]

text_splitter = SemanticChunker(embedding_model)
docs = text_splitter.create_documents(cleaned_pages)

chunked_content = [doc.page_content for doc in docs]

for i in range(3):
  print(f"Chunk {i+1}:\n{chunked_content[i]}\n")

Chunk 1:
The Health Code These are regulations that were formulated to allow the  Department to effectively protect the health of the population. Among the rules embodied in the Health Code is Article 81 which regulates the operations of food establishments for the purpose of preventing public health hazards. Environmental Health Division  The Division of Environmental Health is the Commission within the Health Department that is concerned with public health and works to eliminate the incidence of injury and illness caused by environmental factors. There are several Offices and Bureaus within this division. One of these is the Bureau of Food Safety and Community Sanitation that has the responsibility for conducting inspections of food service and food processing establishments. These inspections are performed by Public Health Sanitarians. Anti-corruption Warning All Sanitarians have Department of Health and Mental Hygiene badges and identification cards which they must display whenever

In [84]:
# prompt: Use the embedding_model to generate embeddings of the text chunks, saving them to a list called chunked_embeddings. To do so, pass your list of chunks to the VertexAIEmbeddings class's embed_documents() method

chunked_embeddings = embedding_model.embed_documents(chunked_content)


In [86]:
# You should have successfully chunked & embedded a short section of the document.
# To get the chunks & corresponding embeddings for the full document, run the following code:

!gcloud storage cp gs://partner-genai-bucket/genai069/chunked_content.pkl .
!gcloud storage cp gs://partner-genai-bucket/genai069/chunked_embeddings.pkl .

chunked_content = pickle.load(open("chunked_content.pkl", "rb"))
chunked_embeddings = pickle.load(open("chunked_embeddings.pkl", "rb"))

# Do not delete this logging statement.
client = google.cloud.logging.Client()
client.setup_logging()
log_message = f"chunked contents are: {chunked_content[0][:20]}"
logging.info(log_message)

Copying gs://partner-genai-bucket/genai069/chunked_content.pkl to file://./chunked_content.pkl
Copying gs://partner-genai-bucket/genai069/chunked_embeddings.pkl to file://./chunked_embeddings.pkl

Average throughput: 109.7MiB/s


INFO:root:chunked contents are: The Health Code Thes


In [ ]:
# Task 3. Prepare your vector database

In [129]:
# prompt: Create a Firestore database with the default name of (default) in Native Mode and leave the other settings to default.

!gcloud firestore databases create \
--location="us-central1"

metadata:
  '@type': type.googleapis.com/google.firestore.admin.v1.CreateDatabaseMetadata
name: projects/qwiklabs-gcp-00-fc042da96977/databases/(default)/operations/IYA7bTAu1LdUTO7b-ufI5BAqMWxhcnRuZWMtc3ULIgUQMYT9kBAGzYKCmQgLCi4a
response:
  '@type': type.googleapis.com/google.firestore.admin.v1.Database
  appEngineIntegrationMode: DISABLED
  concurrencyMode: PESSIMISTIC
  createTime: '2026-02-26T17:21:29.102842Z'
  databaseEdition: STANDARD
  deleteProtectionState: DELETE_PROTECTION_DISABLED
  earliestVersionTime: '2026-02-26T17:21:29.102842Z'
  etag: ILbChc/V95IDMPqThc/V95ID
  freeTier: true
  locationId: us-central1
  name: projects/qwiklabs-gcp-00-fc042da96977/databases/(default)
  pointInTimeRecoveryEnablement: POINT_IN_TIME_RECOVERY_DISABLED
  realtimeUpdatesMode: REALTIME_UPDATES_MODE_ENABLED
  type: FIRESTORE_NATIVE
  uid: e4c8e7fa-dbee-4c54-b7d4-2e306d3b8021
  updateTime: '2026-02-26T17:21:29.102842Z'
  versionRetentionPeriod: 3600s


In [130]:
# prompt: Next, in your Colab Enterprise Notebook populate a db variable with a Firestore Client.

db = firestore.Client()

In [131]:
# prompt: Use a variable called collection to create a reference to a collection named food-safety

collection = db.collection("food-safety")

In [140]:
# prompt: Using a combination of your lists chunked_content and chunked_embeddings, add a document to your collection for each of your chunked documents. Each document can be assigned a random ID, but it should have a field called content to store the chunk text and a field called embedding to store a Firestore Vector() of the associated embedding

for i in range(len(chunked_content)):
  doc_ref = collection.document()
  doc_ref.set({
      "content": chunked_content[i],
      "embedding": Vector(chunked_embeddings[i])
  })


In [ ]:
# Complete the function below to receive a query,
# get its embedding, and compile a context consisting of the text from the 5 documents
# with the most similar embeddings.
# This time, use the embed_query() method
# of the LangChain VertexAIEmbeddings embedding_model
# to embed the user's query

In [156]:
def search_vector_database(query: str):
  context = ""
  # 1. Generate the embedding of the query
  query_embedding = embedding_model.embed_query(query)

  # 2. Get the 5 nearest neighbors from your collection.
  results = collection.find_nearest(
      query_vector=Vector(query_embedding),
      limit=5,
      distance_measure=DistanceMeasure.COSINE,
      vector_field="embedding"
  )

  # Call the get() method on the result of your call to find_nearest to retrieve document snapshots.
  for snapshot in results.get():
    # 3. Call to_dict() on each snapshot to load its data.
    # Combine the snapshots into a single string named
    data = snapshot.to_dict()
    context += data["content"] + "\n"

  return context

In [158]:
search_vector_database("How should I store food?")

FailedPrecondition: 400 Missing vector index configuration. Please create the required index with the following gcloud command: gcloud firestore indexes composite create --project=qwiklabs-gcp-00-fc042da96977 --collection-group=food-safety --query-scope=COLLECTION --field-config=vector-config='{"dimension":"768","flat": "{}"}',field-path=embedding

In [159]:
!gcloud firestore indexes composite create \
--project=qwiklabs-gcp-00-fc042da96977 \
--collection-group=food-safety \
--query-scope=COLLECTION \
--field-config=vector-config='{"dimension":"768","flat": "{}"}',field-path=embedding

Create request issued
Created index [CICAgOjXh4EK].


In [163]:
Markdown(search_vector_database("How should I store food?"))

 Store foods away from dripping condensate , at least six inches above the floor and with enough space between items to encourage air circulation. Freezer Storage Freezing is an excellent method for prolonging the shelf life of foods. By keeping foods frozen solid, the bacterial growth is minimal at best. However, if frozen foods are thawed and then refrozen, then harmful bacteria can reproduce to dangerous levels when thawed for the second time. In addition to that, the quality of the food is also affected. Never refreeze thawed foods, instead use them immediately. Keep the following rules in mind for freezer storage:  Use First In First Out method of stock rotation. All frozen foods should be frozen solid with temperature at 0°F or lower. Always use clean containers that are clearly labeled and marked, and have proper and secure lids. Allow adequate spacing between food containers to allow for proper air circulation. Never use the freezer for cooling hot foods. * * Tip: When receiving multiple items, always store the frozen foods first, then foods that are to be refrigerated, and finally the non perishable dry goods. Dry Storage Proper storage of dry foods such as cereals, flour, rice, starches, spices, canned goods, packaged foods and vegetables that do not require refrigeration ensures that these foods will still be usable when needed. Adequate storage space as well as low humidity (50% or less), and low temperatures (70 °F or less) are strongly recommended.
Only use food containers that are clean, non-absorbent and are made from food-grade material intended for such use. Containers made from metal may react with certain type of high acid foods such as sauerkraut, citrus juices, tomato sauce, etc. Plastic food-grade containers are the best choice for these types of foods. Containers made of copper, brass, tin and galvanized metal should not be used. The use of such products is prohibited. Re-using cardboard containers to store cooked foods is also a source of contamination. Lining containers with newspapers, menus or other publication before placing foods is also prohibited as chemical dyes from these can easily leach into foods. Storage Areas Foods should only be stored in designated areas. Storing foods in passageways, rest rooms, garbage areas, utility rooms, etc. would subject these to contamination. Raw foods must always be stored below and away from cooked foods to avoid cross contamination. Refrigerated Storage This type of storage is typically used for holding potentially hazardous foods as well as perishable foods for short periods of time—a few hours to a few days. An adequate number of efficient refrigerated units are required to store potentially hazardous cold foods. By keeping cold foods cold, the microorganisms that are found naturally on these foods are kept to a minimum. Cold temperature does not kill microorganisms, however, it slows down their growth. Pre-packaged cold foods must be stored at temperatures recommended by the manufacturer. This is especially important when dealing with vacuum packed foods, modified atmosphere packages and sous vide foods. Smoked fish is required by the Health Code to be stored at 38°F or below. Fresh meat, poultry and other potentially hazardous foods must be stored at 41°F or below, while frozen foods must be stored at 0°F or below. For foods to be maintained at these temperatures, refrigerators and freezers must be operating at temperatures lower than 41°F and 0°F., respectively. Thermometers placed in the warmest part of a refrigerated unit are necessary to monitor the temperature of each unit. The rule of storage, First In First Out (FIFO) ensures that older deliveries are used up before newer ones. In practicing FIFO, the very first step would be to date all products as they are received. The next step is to store the newer products behind the older ones. The following rules are important in making sure that foods are safe during refrigerated storage:  Store cooked foods above raw foods to avoid cross-contamination. Keep cooked food items covered unless they are in the process of cooling, in which case they must be covered after being cooled to 41°F. Avoid placing large pots of hot foods in a refrigerator.
l Store food in vermin-proof containers — metal or glass  containers, with tightly fitted lids. l Remove dented, leaking, rusted, swollen or unlabeled canned goods. Cold Storage: l All PHFs must be stored at 41° F (Except smoked fish at 38° F and raw shell eggs at 45 ° F). l All cooked and ready-to-eat food must be stored away from and above raw food. l Do not store foods in quantities that exceed the storage unit’s  capacity. l Place a refrigeration thermometer in the warmest spot in the unit to measure ambient air temperature of the unit l Check for condensation that may contaminate food. l Keep frozen foods frozen at 0° F or lower. STORAGE
Furthermore, it is improper to store food in ice machines or ice that will be later used for human consumption. Food should be stored at least six inches off the floor, away from walls and dripping pipes. Keep all food, bulk or otherwise, covered and safe from contamination. Check food daily and throw away any spoiled or contaminated food. Store cleaning, disinfecting, and other chemicals away from foods, clearly marked and in their original containers. Keep food refrigerated at a temperature of 41°F or below. Monitor temperatures regularly with a thermometer placed in the warmest part of the refrigerator. Keep all cooling compartments closed except when you are using them. Store food in a refrigerator in such a way that the air inside can circulate freely. Keep all refrigerated foods covered, and use up stored leftovers quickly. When dishes and utensils are sparkling clean, keep them that way by proper storage.
In addition to the above, avoid sunlight as it may affect the quality of some foods. Following are some of the guidelines:  Use First In First Out method of stock rotation. Keep foods at least 6 inches off the floor. This allows for proper cleaning and to detect vermin activity. Keep foods in containers with tightly fitted lids. Keep dry storage areas well lighted and ventilated. Install shades on windows to prevent exposure from sunlight. Do not store foods under overhead water lines that may drip due to leaks or condensation. Do not store garbage in dry food storage areas. Make sure that dry storage area is vermin proof by sealing walls and baseboards and by repairing holes and other openings. * * Safety Tip: Storage of harmful chemicals in the food storage areas can create hazardous situations and hence is prohibited by law. All chemicals must be labeled properly and used in accordance to the instructions on the label. Pesticide use is prohibited unless used by a licensed pest control officer. Storage in Ice Whenever food items are to be stored in ice, care must be taken to ensure that water from the melted ice is constantly being drained so that the food remains on ice and not immersed in iced water.
